# 📗 가설검정·회귀 — 가설검정

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

지난 시간엔 표본으로 전체를 **추측**하고(신뢰구간), 신뢰구간으로 **어떤 주장이 데이터와 맞는지** 판단해 봤습니다. 이번 시간엔 그 판단을 **귀무가설·대립가설·p-value 를 갖춘 정식 가설검정**으로 옮깁니다. "두 그룹 평균이 정말 다른가?", "세 집단은?", "두 범주가 관련 있나?"를 **t-검정·분산분석(ANOVA)·카이제곱·비모수 검정**으로 판정하고, 차이의 크기를 **효과크기**로 잽니다.

## ⏪ 복습 — 지난 시간: 통계적 추론(표본분포·CLT·신뢰구간)

지난 시간에 쌓은 것들이 이번 시간의 **재료**가 됩니다. 오늘은 여기에 **가설검정**의 틀을 새로 더합니다.

- **신뢰구간으로 주장 판단 → 정식 가설검정**: 지난 시간엔 "신뢰구간이 주장값을 포함하는가"로 주장이 데이터와 맞는지 판단했습니다. 이번 시간엔 같은 판단을 **귀무가설을 세우고 p-value 로** 정식화합니다 — 사실 둘은 **동전의 양면**입니다.
- **분포 → 검정통계량의 분포**: 지난 시간의 정규분포·표본평균 분포처럼, 각 검정은 **검정통계량이 (귀무가설 아래에서) 따르는 분포**(t분포·F분포·χ²분포)를 가집니다. **p-value** 는 그 분포에서 관측값보다 바깥쪽 꼬리 넓이인데, 이 개념은 **오늘 새로** 배웁니다.
- **CLT → 검정의 강건성**: 중심극한정리 덕분에 **표본이 크면**(각 그룹 대략 n≥30) 표본평균은 정규에 가까워집니다. 그래서 t-검정·ANOVA는 원자료가 다소 비정규여도 **대표본에선 꽤 강건**합니다. 이 점이 뒤의 '가정 점검'에서 중요합니다.

**오늘의 목표**

- [ ] **가설검정 프레임**(H₀/H₁·유의수준 α·검정통계량·p-value·1종/2종 오류)을 말로 설명한다.
- [ ] **가정 점검** — 정규성(`pg.normality`·Q-Q Plot)과 등분산(`pg.homoscedasticity`)을 확인하고 검정을 고른다.
- [ ] **t-검정 3종**(1표본·대응·독립)을 `pg.ttest` 로 쓰고 **Cohen's d** 로 효과크기를 잰다.
- [ ] 정규성이 깨지면 **비모수 검정**으로 갈아탄다 — 독립 2집단 `pg.mwu`, 대응 `pg.wilcoxon`, 3집단+ `pg.kruskal`.
- [ ] 3집단 이상은 **ANOVA**(`pg.anova`) + **η²** + 사후검정(**Tukey HSD**, `pg.pairwise_tukey`)으로 분석한다.
- [ ] 범주형은 **카이제곱**(적합도·독립성) + **Cramér's V** 로 검정한다.
- [ ] 상황에 맞는 검정을 **의사결정 트리**로 스스로 고른다.

In [ ]:
# [제공 코드] 통계 검정에 쓸 라이브러리와 한글 폰트를 준비합니다.
import warnings
warnings.filterwarnings('ignore')   # pingouin 의 사소한 경고를 숨겨 출력을 깔끔하게
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg               # ★ 이번 단원 주력: 검정+효과크기+신뢰구간을 한 번에
from scipy import stats             # 일부 보조(적합도·개념 데모)에 사용

import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지
sns.set_theme(font=KOREAN_FONT, rc={'axes.unicode_minus': False})

## 🔧 이번 단원의 주력 도구 — Pingouin

가설검정부터는 **Pingouin**(`pingouin`, 줄여서 `pg`)을 주력으로 씁니다. Pingouin은 **통계 검정 전용** 라이브러리로, 한 번 호출하면 **검정통계량 · p-value · 신뢰구간(CI) · 효과크기 · 검정력**을 **표(DataFrame) 하나**로 돌려줍니다.

지금까지 쓰던 `scipy.stats`는 검정통계량과 p-value만 주어서 **효과크기(Cohen's d 등)를 매번 손으로** 계산해야 했습니다. Pingouin은 그 효과크기를 **기본으로 포함**합니다 — 그래서 "유의성(p)만 보고 **효과크기를 빼먹는** 실수"를 구조적으로 막아 줍니다. 결과가 표라서 그대로 저장·재사용하기도 좋습니다.

```python
pip install pingouin      # 설치 (실습 환경엔 이미 준비되어 있습니다)
import pingouin as pg
```

### Pingouin ↔ 기존 라이브러리(대안) 대응표
| 하는 일 | Pingouin (주력) | 기존 방법(대안) |
|---|---|---|
| 정규성 점검 | `pg.normality` | `scipy.stats.shapiro` |
| 등분산 점검 | `pg.homoscedasticity` | `scipy.stats.levene` |
| t-검정(+Cohen's d) | `pg.ttest` | `scipy.stats.ttest_*` + 수동 d |
| 비모수 2집단(+효과크기) | `pg.mwu` | `scipy.stats.mannwhitneyu` |
| ANOVA(+η²) | `pg.anova` | `scipy.stats.f_oneway` + 수동 η² |
| 사후검정(+효과크기) | `pg.pairwise_tukey` | `statsmodels`의 `pairwise_tukeyhsd` |
| 카이제곱 독립성(+Cramér's V) | `pg.chi2_independence` | `scipy.stats.chi2_contingency` + 수동 V |
| 상관(+신뢰구간) | `pg.corr` | `scipy.stats.pearsonr` |
| **회귀분석** | (미지원) | **`statsmodels`의 `smf.ols`** — 다음 시간 |

> **Pingouin이 못 하는 것은 원래 도구로** 갑니다: **회귀는 `statsmodels`**, **카이제곱 적합도·조정된 잔차**는 `scipy`·`statsmodels`. 코드에서 그럴 때마다 🔧 표시로 짚어 드립니다.

In [ ]:
# Pingouin 미리보기 — 딱 한 줄이 검정+효과크기+신뢰구간+검정력을 한 표로 준다
demo_a = pd.Series([12, 15, 14, 10, 13, 16, 11])
demo_b = pd.Series([18, 20, 17, 19, 22, 16, 21])
display(pg.ttest(demo_a, demo_b))   # 검정통계량·자유도·p값·신뢰구간·효과크기·검정력을 한 번에

# 표에 뜬 8개 열이 각각 무엇인지 — 하나도 빠짐없이
print('T           = 검정통계량 (차이를 표준화한 값. 0에서 멀수록 드문 결과)')
print('dof         = 자유도 (분포의 모양을 정하는 값)')
print('alternative = 대립가설의 방향 (two-sided = 양측검정, 기본값 — 의도한 검정이 맞는지 확인)')
print('p_val       = p값 (0.05 보다 작으면 유의)')
print('CI95        = 95% 신뢰구간 (평균 차이의 범위. 0을 안 품으면 유의와 같은 뜻)')
print('cohen_d     = 효과크기 (0.2 작음 / 0.5 중간 / 0.8 큼)')
print('power       = 관측 검정력 (지금 관측된 효과크기를 기준으로 계산 — p 가 작으면 자동으로 커집니다)')
print('BF10        = 베이즈 인자 (다른 통계 학파의 지표 — 이 단원에서는 보지 않습니다)')
print()
print('→ 열이 많아 보이지만 결론에 쓰는 것은 몇 개뿐입니다. 다음 절에서 정리합니다.')

## 📊 결과 표 읽는 법 — 무엇을 보고 무엇을 무시할까

Pingouin 은 결과를 **표(DataFrame)** 로 주는데 열이 많습니다. 하지만 **전부 볼 필요가 없습니다.** 실제로 결론을 만드는 건 **딱 세 가지**뿐입니다. (값을 꺼낼 땐 `결과['열이름'].iloc[0]`.)

> ### 🧭 결과를 결론으로 바꾸는 3단계 — **이 세 개만 보면 됩니다**
> 1. **⭐ 유의한가?** → **p-value** 를 α(0.05)와 비교 → "우연이라 보기 어렵다/어렵지 않다"
> 2. **⭐ 얼마나 큰가?** → **효과크기** 를 해석 기준과 비교 → "작다/중간/크다"
> 3. **⭐ 어느 방향인가?** → 평균·비율 등 **실제 값** → "어느 쪽이 더 높다"
>
> **p 가 작아도 효과크기가 작으면 "통계적으로는 유의하지만 실질적 의미는 작다"** 가 정답입니다. 표본이 크면 사소한 차이도 p 가 작아지기 때문입니다. 나머지 열은 **참고**이거나 **이 단원에선 안 봐도 되는** 것들입니다.

### ⭐ 꼭 봐야 할 열 — 결론을 만드는 것
| 열 | 무엇 | 읽는 법 |
|---|---|---|
| `p_val` · `p_unc` · `pval` · `p_tukey` | **p-value** | **이름만 다를 뿐 전부 같은 뜻.** `< 0.05` 면 유의 |
| `cohen_d` | 효과크기(t-검정) | **0.2 작음 / 0.5 중간 / 0.8 큼** (부호 없는 크기 \|d\|) |
| `np2` | 효과크기 **η²**(ANOVA) | 집단 구분이 전체 변동의 몇 %를 설명. **0.01 / 0.06 / 0.14** |
| `cramer` | 효과크기 **Cramér's V**(카이제곱) | 두 범주의 연관 세기. **0.1 / 0.3 / 0.5** |
| `RBC` · `hedges` | 효과크기(비모수 · 사후검정) | 기준은 위와 같음(0.1/0.3/0.5, 0.2/0.5/0.8) |
| `normal` · `equal_var` | 가정 점검 결과 | **True 면** 정규/등분산 가정 **만족** (= `pval` > 0.05) |
| `mean_A` · `mean_B` · `diff` | 집단별 실제 값 | 위 **3단계의 ③ 어느 방향인가**를 판정하는 값. 어느 쪽이 큰지는 이 값으로 말합니다 |

### ○ 참고로만 보는 열 — 몰라도 결론은 낼 수 있음
| 열 | 뜻 |
|---|---|
| `T` · `F` · `U_val` · `chi2` · `W` | **검정통계량** — 차이를 표준화한 값. p-value 를 만드는 재료(0에서 멀수록 드묾). 보고서엔 관례상 함께 씁니다 |
| `dof` · `DF` · `ddof1`·`ddof2` | 자유도 — 분포의 모양을 정하는 값 |
| `CI95` | 95% 신뢰구간 — 참값이 있을 법한 범위(평균 차이 구간이 **0을 안 품으면** 유의와 같은 뜻) |
| `power` | **관측 검정력** — **지금 관측된 효과크기가 참이라 가정**했을 때의 1−β. 설계 단계의 검정력(보통 0.80으로 미리 정하는 값)과는 **다른 값**이고, **p 가 정해지면 따라 정해져** 새 정보가 없습니다 |
| `CLES` | 한쪽에서 뽑은 값이 다른 쪽보다 클 확률(직관적 표현) |
| `alternative` | **내가 의도한 검정인지 반드시 확인할 설정값.** `two-sided`(기본)=양측, `greater`·`less`=단측. 단측은 p 가 양측의 **절반**이라 결론이 뒤집힐 수 있습니다 — 같은 데이터에서 양측 p=0.0002(유의)가 방향을 반대로 잡은 단측에서는 p=0.9999(유의하지 않음)가 됩니다 |

### ✕ 이 단원에선 **안 봐도 되는** 열 — 무시하세요
| 열 | 왜 무시해도 되나 |
|---|---|
| `BF10` | **베이즈 인자** — 이 과정에서 다루지 않는 다른 통계 학파(베이즈)의 지표입니다. 눈에 보여도 그냥 넘기세요 |
| `SS` · `MS` (ANOVA 표) | F 와 η² 를 만드는 **중간 계산값**입니다. 원리 설명용이니 결론엔 쓰지 않습니다 |
| `se` | 표준오차 — 신뢰구간·검정통계량을 만드는 중간 계산값입니다 |

> 표에 낯선 열이 보여도 **당황하지 마세요** — 결론은 위 ⭐ 세 가지(유의성·효과크기·방향)로 냅니다. 다만 `alternative` 만은 **내가 의도한 검정(양측/단측)이 맞는지 한 번 확인**하고 넘어가세요 — 여기가 어긋나면 위 세 가지를 아무리 잘 읽어도 결론이 틀립니다.

## 데이터 살펴보기 — 자동차 연비(mpg) + 훈련 기록(before/after)

이번 시간엔 두 데이터를 씁니다.

- **`mpg`** (398대): 연비·무게·마력·제조국 등. 집단 비교(제조국별 연비)·범주 검정(제조국 분포)에 씁니다.
- **`train_before_after`** (12명): 같은 사람의 훈련 **전(before_sec)·후(after_sec)** 기록. **대응표본** t-검정에 씁니다.

새 데이터를 만나면 분석 전에 **생김새부터** 봅니다 — 앞부분(`head`)·구조와 결측(`info`)·수치 요약(`describe`).

In [ ]:
mpg = pd.read_csv('data/mpg.csv')
train = pd.read_csv('data/train_before_after.csv')

print('mpg 크기:', mpg.shape, ' / train 크기:', train.shape)
print('\n[mpg 앞부분]')
display(mpg.head())
print('\n[mpg 구조와 결측]')
mpg.info()
print('\n[mpg 수치형 요약]')
display(mpg.describe())
print('\n[제조국(origin) 분포]')
display(mpg['origin'].value_counts().to_frame('대수'))
print('\n[train 대응표본 — 훈련 전/후 기록(초)]')
display(train.head())

---
# 1. 가설검정 프레임 — 우연인가, 진짜인가

## 왜 필요할까요?
표본에서 "차이가 있어 보인다"고 다 진짜는 아닙니다. **우연히** 그만큼 차이가 날 수도 있으니까요. 가설검정은 "이 차이가 **우연이라고 보기엔 너무 드문가**"를 확률로 판정하는 **정해진 절차**입니다.

<img src="images/가설검정_프로세스.png" width="720">

### 절차와 용어
1. **가설 세우기** — 귀무가설 H₀("차이 없다·효과 없다", 기본 입장)와 대립가설 H₁("차이 있다", 보이고 싶은 주장).
2. **유의수준 α 정하기** — 우연을 진짜로 오해할 위험의 상한. 보통 **0.05**(5%).
3. **검정통계량 계산** — 관측된 차이를 표준화한 값(t, F, χ² 등). 클수록 "우연치고는 드문" 쪽.
4. **p-value 구하기** — H₀가 참일 때 **이만큼 또는 더 극단적인** 통계량이 나올 확률(귀무분포의 꼬리 넓이).
5. **판정** — `p < α` 면 H₀ **기각**("유의하다"), `p ≥ α` 면 **기각 못함**("근거 부족").

> **주의: '기각 못함' ≠ 'H₀가 참'.** 증거가 부족할 뿐입니다. 재판의 무죄추정과 같습니다 — "유죄를 입증 못 함"이 "결백 증명"은 아니듯이.

## p-value 를 귀무분포에서 읽기

**p-value** 는 이번 단원에서 처음 배우는 개념입니다: **귀무가설이 참이라면** 검정통계량이 관측값만큼 또는 그보다 더 극단적으로 나올 확률 — 즉 **검정통계량의 귀무분포에서 관측값보다 바깥쪽 넓이**입니다. 아래 그림에서 빨간 넓이가 바로 그 확률입니다. 검정통계량이 클수록(중앙에서 멀수록) 꼬리 넓이=p 가 작아집니다.

<img src="images/p_value_개념.png" width="780" style="max-width:100%">

In [ ]:
# 검정통계량 → 귀무분포 → p-value: 관측 통계량이 우연히 나올 꼬리 넓이
z_obs = 2.1
p_two_sided = 2 * stats.norm.sf(abs(z_obs))   # 양측: 양쪽 꼬리
print('관측 검정통계량 z =', z_obs)
print('양측 p-value      =', round(p_two_sided, 4), '  (|z|이 2.1보다 클 확률)')

alpha = 0.05
decision = '기각(유의하다)' if p_two_sided < alpha else '기각 못함(근거 부족)'
print('유의수준 α        =', alpha, '  (실험 전에 정한 기준선)')
print('판정: p < α ?      =', p_two_sided < alpha, '→', decision)

grid = np.linspace(-4, 4, 400)
density = stats.norm.pdf(grid)
fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(x=grid, y=density, color='black', ax=ax)          # 귀무분포 곡선
# 꼬리 색칠은 seaborn 에 대응 함수가 없어 matplotlib 의 fill_between 을 씁니다
ax.fill_between(grid, density, where=(np.abs(grid) >= z_obs), color='tomato')
ax.axvline(z_obs, color='red', linestyle='--')
ax.axvline(-z_obs, color='red', linestyle='--')
ax.set_title('귀무분포와 p-value — 귀무가설이 참일 때 이만큼 극단적일 확률')
ax.set_xlabel('검정통계량'); ax.set_ylabel('밀도')
plt.show()

## p-value vs 유의수준(α) — 헷갈리지 말자

둘 다 0~1 사이 확률이라 헷갈리기 쉽지만, **역할이 완전히 다릅니다.**

| | 유의수준 α | p-value |
|---|---|---|
| **언제 정하나** | 실험 **전에** 연구자가 **선택** | 데이터를 본 **후에** 계산 |
| **무엇인가** | 감수할 1종 오류의 **상한(기준선)** | 관측이 H₀ 하에서 이만큼 극단적일 **확률(증거)** |
| **값** | 고정(보통 0.05) | 데이터마다 달라짐 |
| **역할** | 판정의 **커트라인** | 그 커트라인과 **비교되는 관측값** |

- **핵심 한 줄**: α는 **미리 그어 둔 선**, p-value 는 **관측이 그 선의 어느 쪽에 떨어지는지**입니다. 관측이 기준보다 더 드물면(**p < α**) 귀무가설을 기각합니다.
- **비유**: α 는 "우연이라고 보기엔 너무 드물다"고 인정할 **기준 확률**(예: 5%)을 시험 전에 정해 둔 것이고, p-value 는 관측이 **실제로 얼마나 드문지**를 데이터에서 잰 값입니다. 잰 값이 기준선 아래면 "우연 아님"으로 판정합니다.

> **흔한 오해 3가지** — p-value 는 아래가 **아닙니다**:
> - ❌ "귀무가설이 참일 확률" — p 는 H₀가 참일 확률이 아닙니다.
> - ❌ "결과가 순전히 우연일 확률" — 정확히는 **H₀가 참이라고 가정했을 때** 관측이 이만큼 극단적일 확률입니다.
> - ❌ "효과의 크기" — p 가 작다고 효과가 큰 건 아닙니다. 표본이 크면 **사소한 차이도** p 가 작아집니다(크기는 **효과크기**가 말해 줍니다).

## 양측 검정과 단측 검정 — 꼬리를 몇 쪽 세는가

**대립가설 H₁ 이 방향을 갖는지**에 따라 갈립니다. 이 선택이 p-value 를 **두 배로 바꾸므로** 결론이 뒤집힐 수 있습니다.

| | 대립가설 H₁ | 보는 꼬리 | 언제 |
|---|---|---|---|
| **양측 검정** (기본) | "다르다" (≠) | **양쪽** — α 를 절반씩 나눠 둠 | **방향을 모를 때**. 안전한 기본값 |
| **단측 검정** | "더 크다"(>) 또는 "더 작다"(<) | **한쪽** — α 를 한쪽에 몰아 둠 | 방향을 **미리 확신할 근거**가 있을 때 |

- 관측된 차이가 H₁ 이 말한 **그 방향이면 단측 p 는 양측 p 의 정확히 절반**입니다.
- 반대로 **방향을 잘못 잡으면** 단측 p 가 1 에 가까워져 유의하지 않게 됩니다.
- **방향은 데이터를 보기 전에 정합니다.** 결과를 보고 유리한 쪽으로 바꾸면 1종 오류가 부풀어요.

> Pingouin 은 **기본이 양측**(`alternative='two-sided'`)입니다. 결과 표의 `alternative` 열로 **내가 의도한 검정이 맞는지 확인**하세요 — 이 단원에서는 특별한 말이 없으면 **양측**을 씁니다.

## 1종 오류와 2종 오류 — 두 가지 방식으로 틀린다

판정은 두 방향으로 틀릴 수 있습니다.

| 실제 \ 판정 | H₀ 기각("차이 있다") | 기각 못함("차이 없다") |
|---|---|---|
| **H₀ 참**(진짜 차이 없음) | **1종 오류(α)** — 없는 걸 있다고 | 옳은 판정 |
| **H₀ 거짓**(진짜 차이 있음) | 옳은 판정(검정력 1−β) | **2종 오류(β)** — 있는 걸 놓침 |

<img src="images/1종_2종_오류.png" width="600">

- **1종 오류(α)**: 실제로 효과가 없는데 "있다"고 결론(거짓 양성). α=0.05 로 이 위험을 통제합니다.
- **2종 오류(β)**: 실제로 효과가 있는데 "없다"고 놓침(거짓 음성). **검정력** = 1−β.
- α를 낮추면 1종 오류는 줄지만 2종 오류가 늘어 **맞바꿈(trade-off)** 관계입니다. 표본을 키우면 둘 다 줄일 수 있습니다.

### 🖐️ 함께 따라하기 — 검정통계량에서 p-value 구하기

관측 검정통계량이 `z_obs = 2.0` 일 때, 표준정규 귀무분포에서 **양측 p-value** 를 직접 구해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) z_obs = 2.0 이라고 하자
# 2) 양측 p-value = 2 * stats.norm.sf(abs(z_obs)) 를 소수 넷째 자리로 출력한다
# 3) p 가 0.05보다 작은지(유의한지) True/False 로 출력한다

### ✅ 바로 확인 퀴즈

**1.** p-value = 0.03 이고 유의수준 α = 0.05 입니다. 귀무가설을 어떻게 하나요?

<details><summary>정답 보기</summary>

`p < α` 이므로 귀무가설을 **기각**합니다("유의하다"). 관측된 차이가 우연이라고 보기엔 드물다는 뜻입니다.

</details>

**2.** 실제로는 효과가 있는데 검정이 "차이 없다"고 결론했습니다. 무슨 오류인가요?

<details><summary>정답 보기</summary>

**2종 오류(β)** 입니다(있는 효과를 놓친 거짓 음성). 반대로 없는 효과를 "있다"고 하면 1종 오류(α)입니다. 검정력은 1−β 로, 표본이 클수록 커집니다.

</details>

**3.** "p ≥ α 라서 기각 못함" 은 "귀무가설이 참임을 증명했다" 와 같은 말인가요?

<details><summary>정답 보기</summary>

아니요. **증거가 부족할 뿐**, 참을 증명한 것이 아닙니다. 표본이 작아 검정력이 낮아서 못 잡았을 수도 있습니다. (재판의 "무죄추정"과 같습니다 — 유죄 입증 실패 ≠ 결백 증명.)

</details>

---
# 2. 가정 점검 — 어떤 검정을 쓸 수 있나

## 왜 필요할까요?
t-검정·ANOVA 같은 **모수 검정**은 **세 가지 전제** 위에 섭니다 — **독립성**(관측치끼리 서로 영향을 주지 않음) · **정규성**(각 집단이 대략 정규분포) · **등분산**(집단끼리 퍼진 정도가 비슷함). 가정이 깨지면 p-value 를 믿기 어려우니 검정 전에 확인합니다.

> **독립성은 검정으로 확인하는 것이 아닙니다.** `pg.normality`·`pg.homoscedasticity` 같은 도구가 없어요. **자료를 어떻게 모았는지**로 판단합니다 — 서로 다른 사람에게서 한 번씩 얻었다면 독립이고, **같은 대상을 두 번 잰 자료라면 독립이 아니라 대응표본**입니다(3절에서 대응표본 t 를 따로 쓰는 이유). 그래서 아래에서는 **검정으로 확인할 수 있는 두 가지**, 정규성과 등분산을 봅니다.

### ① 정규성 — Pingouin `pg.normality` + Q-Q Plot
- **`pg.normality(x)`** → 표로 `W`(Shapiro-Wilk 통계량)·`pval`·`normal`(정규 여부 True/False)을 줍니다. H₀ 는 "정규분포다". 그래서 **`pval > 0.05`(=`normal` True)면 정규성을 기각 못함**(정규로 봐도 무방), 작으면 비정규.
- **Q-Q Plot(`pg.qqplot`)** — 데이터 분위수를 정규 분위수와 견주는 그림. 점이 **직선을 따르면** 정규에 가깝습니다. 아래 6가지 패턴으로 읽습니다.

<img src="images/QQ_Plot_패턴_가이드.png" width="720">

> 🔧 **대안**: `pg.normality` 는 내부적으로 `scipy.stats.shapiro` 와 같은 Shapiro-Wilk 검정입니다 — 값은 같고 결과를 표로 정리해 줄 뿐입니다. Q-Q Plot 도 `scipy.stats.probplot` 으로 그릴 수 있습니다.

> **Shapiro 역설 & Q-Q 우선**: Shapiro 는 표본이 크면(n>50) 사소한 이탈에도 p가 작아져 "비정규"라 외치고, 작으면(n<20) 둔감합니다. 그래서 수치 하나로 단정하지 말고 **Q-Q Plot 을 함께 보고, 둘이 엇갈리면 Q-Q Plot 을 우선**합니다.

### ② 등분산 — Pingouin `pg.homoscedasticity`
- **`pg.homoscedasticity(data, dv, group)`** → `W`·`pval`·`equal_var`(등분산 여부). H₀ 는 "분산이 모두 같다". **`pval > 0.05`(=`equal_var` True)면 등분산 가정 만족**. (기본은 Levene 검정 — 🔧 대안 `scipy.stats.levene` 와 동일.)
- 이 함수는 **긴 형태(long-form)** 데이터를 받습니다: 값 열(`dv`)과 그룹 열(`group`)을 가진 하나의 DataFrame. 특정 두 그룹만 보려면 `data=df[df['group'].isin(['a','b'])]` 로 걸러 넣습니다.
- 독립 2표본에서 **등분산이면 Student's t**, **위반이면 Welch's t** 로 갈립니다(3절). 대응표본은 등분산이 필요 없습니다(차이값만 봄).

> **CLT 덕분에**: 각 집단 표본이 충분히 크면(대략 n≥30) 표본평균은 정규에 가까워져, 원자료가 다소 비정규여도 t·ANOVA는 **꽤 강건**합니다. Shapiro 는 **원자료**를 보지만, 검정은 사실 **표본평균의 분포**를 씁니다 — 그래서 대표본에선 정규성 위반에 관대할 수 있습니다.

In [ ]:
# ① 정규성: pg.normality (pval>0.05, normal=True 면 정규성을 기각 못함)
norm_result = pg.normality(mpg[['mpg', 'acceleration']])   # 여러 열을 한 번에
display(norm_result)
for col in ['mpg', 'acceleration']:
    row = norm_result.loc[col]
    verdict = '정규성 기각 못함(정규로 무방)' if row['normal'] else '정규성 기각(비정규)'
    print(f'{col:13s}: W={row["W"]:.4f}, p={row["pval"]:.4f} → {verdict}')

# Q-Q Plot(pg.qqplot): 점이 직선을 따르면 정규에 가깝다 (R²도 함께 표시)
fig, ax = plt.subplots(figsize=(6, 5))
pg.qqplot(mpg['mpg'], dist='norm', ax=ax)
ax.set_title('연비(mpg) Q-Q Plot — 오른쪽 꼬리가 위로 휨')
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
pg.qqplot(mpg['acceleration'], dist='norm', ax=ax)
ax.set_title('가속(acceleration) Q-Q Plot — 대체로 직선')
plt.show()

print('가속(acceleration): p 가 0.05를 근소하게 밑돌지만 Q-Q 는 대체로 직선 — 표본이 크면 Shapiro 가')
print('사소한 이탈에도 민감(Shapiro 역설). 이럴 땐 Q-Q Plot 을 우선 본다.')

In [ ]:
# ② 등분산: pg.homoscedasticity — 일본산 vs 유럽산 연비의 분산이 같은가
#    긴 형태(dv=값, group=그룹)로 넣는다. 두 그룹만 보려면 isin 으로 먼저 거른다.
two_origin = mpg[mpg['origin'].isin(['japan', 'europe'])]
lev = pg.homoscedasticity(data=two_origin, dv='mpg', group='origin')
display(lev)
print('→ equal_var =', bool(lev['equal_var'].iloc[0]),
      '(True 면 등분산 → Student t, False 면 이분산 → Welch t)')

### 🖐️ 함께 따라하기 — 대기시간의 정규성 점검

데모는 **자동차 연비(mpg)** 로 봤습니다. 따라하기는 **다른 도메인 — 어느 고객센터의 상담 기록**(`callcenter_calls.csv`)으로 같은 검정을 연습합니다. 상담 600건마다 채널·대기시간·상담시간·만족도 등을 적어 둔 표예요.

고객이 응대까지 기다린 **대기시간(`대기시간_초`)** 이 정규분포를 따르는지 Shapiro 검정과 Q-Q Plot 으로 판단해 봅니다.

> ⚠️ 이 셀에서 만드는 `cc` 를 **이후 따라하기에서 계속 씁니다** — 건너뛰지 말고 꼭 실행하고 넘어가세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# ※ 데모는 '자동차 연비'였죠. 이번엔 다른 데이터(고객센터 상담 기록)로 연습합니다.
# 1) pd.read_csv 로 data/callcenter_calls.csv 를 읽어 cc 에 담고 head()·info() 로 훑어본다
# 2) pg.normality(cc['대기시간_초']) 로 결과 표를 display 한다
# 3) 그 표에서 W(['W'])와 p(['pval'])를 꺼내 출력한다 (p는 지수표기 format(p, '.2e'))
# 4) fig, ax = plt.subplots(figsize=(6,5)) 후 pg.qqplot(cc['대기시간_초'], dist='norm', ax=ax)
# 5) ax.set_title('대기시간 Q-Q Plot') 를 달고 plt.show()
# 6) normal 이 False(p<0.05)이면 정규성 기각 — Q-Q 점들이 직선에서 어떻게 벗어나는지 눈으로도 확인한다

### ✅ 바로 확인 퀴즈

**1.** `pg.normality` 결과가 `pval = 0.24`, `normal = True` 였습니다(α=0.05). 정규성에 대해 어떻게 판단하나요?

<details><summary>정답 보기</summary>

H₀ 는 "정규분포다" 입니다. `pval > 0.05`(`normal=True`) 이므로 **정규성을 기각하지 못합니다** — 정규로 봐도 무방합니다.

</details>

**2.** `pg.homoscedasticity` 결과가 `pval = 0.68`, `equal_var = True`. 독립 2표본 t-검정으로 Student 와 Welch 중 무엇이 적절한가요?

<details><summary>정답 보기</summary>

`pval > 0.05`(`equal_var=True`) 라 **등분산 가정을 기각 못함** → 등분산을 가정하는 **Student's t**(`pg.ttest(..., correction=False)`)가 적절합니다. 만약 p가 작아 등분산이 깨지면 **Welch's t**(`correction=True`)를 씁니다.

</details>

---
# 3. t-검정 3종 — 평균을 비교한다

## 왜 필요할까요?
가장 흔한 질문은 "**평균이 다른가**" 입니다. 상황에 따라 세 종류의 t-검정을 씁니다.

| 종류 | 언제 | Pingouin 함수 | 가정 |
|---|---|---|---|
| **1표본** t | 한 집단 평균이 **특정 기준값**과 다른가 | `pg.ttest(x, 기준값)` | x 정규성 |
| **대응표본** t | **같은 대상**의 전/후(짝지은 두 값) 차이 | `pg.ttest(after, before, paired=True)` | **차이값** 정규성 |
| **독립 2표본** t | **서로 다른** 두 집단의 평균 차이 | `pg.ttest(a, b, correction=)` | 각 집단 정규성 + 등분산 여부 |

> **세 가지가 모두 `pg.ttest` 하나**입니다 — 인자만 바뀝니다(스칼라 기준값=1표본, `paired=True`=대응, `correction`=등분산 선택). 그리고 결과 표에 **Cohen's d(`cohen_d`)와 신뢰구간·검정력이 이미 들어 있습니다.**
> 🔧 **대안(기존)**: `scipy.stats.ttest_1samp` / `ttest_rel` / `ttest_ind(a, b, equal_var=)` — 이쪽은 t·p만 주므로 효과크기는 따로 계산해야 합니다.

> **대응표본의 핵심**: 같은 사람을 전·후로 재면 개인차가 상쇄됩니다. 그래서 등분산이 아니라 **차이값(after−before)의 정규성**만 확인합니다.

### 효과크기 — p-value 만으로는 부족하다
p-value 는 **표본 크기에 좌우**됩니다(n이 크면 사소한 차이도 유의). 그래서 "**얼마나** 큰 차이인가"는 **효과크기**로 따로 잽니다. Pingouin 은 `pg.ttest` 결과의 **`cohen_d` 열로 이 값을 자동으로** 줍니다(원리는 아래).

- **Cohen's d** (독립 2표본): `d = (m1 − m2) / 합동표준편차`, 합동표준편차 `= √(((n₁−1)s₁² + (n₂−1)s₂²)/(n₁+n₂−2))`
- **Cohen's d** (1표본): `d = (평균 − 기준값) / 표준편차`. **대응표본**은 Pingouin 이 **두 조건의 표준편차를 함께 쓴 방식**(`d_av`)으로 `cohen_d` 를 계산합니다.
- 해석 기준: |d| ≈ **0.2 작음 / 0.5 중간 / 0.8 큼**. (Pingouin 의 `cohen_d` 는 부호 없는 크기 |d| 입니다.)

> **참고**: 대응표본 효과크기는 관례가 둘입니다 — 차이값 기준 `d_z = 차이.mean()/차이.std` 와 두 조건 표준편차 기준 `d_av`. **Pingouin 의 `cohen_d` 는 `d_av`** 라서, 손으로 `d_z` 를 구한 값과는 다를 수 있습니다(둘 다 맞는 정의).

<img src="images/효과크기_해석.png" width="720">

In [ ]:
# ① 1표본 t-검정: 이 데이터의 평균 연비가 25(mpg)라는 주장을 검정
#    pg.ttest(x, 기준값) — 결과 표에 t·p·신뢰구간·Cohen's d 가 한 번에 나온다
res1 = pg.ttest(mpg['mpg'], 25)
display(res1)
print('표본 평균 연비 =', round(mpg['mpg'].mean(), 3))
print('H0: 평균=25 →  t =', round(res1['T'].iloc[0], 3), ', p =', format(res1['p_val'].iloc[0], '.4f'),
      ", Cohen's d =", round(res1['cohen_d'].iloc[0], 3))
print('  p<0.05 → 기각: 평균은 25가 아니다 (유의)')

# 같은 검정, 기준값만 23으로 바꾸면? — 기각 못하는 예
res1b = pg.ttest(mpg['mpg'], 23)
print('\nH0: 평균=23 →  t =', round(res1b['T'].iloc[0], 3), ', p =', round(res1b['p_val'].iloc[0], 4))
print('  p>0.05 → 기각 못함: 평균이 23이 아니라고 할 근거는 부족 (비유의)')

In [ ]:
# ② 대응표본 t-검정: 같은 사람의 훈련 전(before)·후(after) 기록 비교
before = train['before_sec']
after = train['after_sec']
diff = after - before                      # 후 - 전 (음수면 기록 단축)

# 대응표본은 '차이값'의 정규성만 확인 (등분산 불필요)
diff_norm = pg.normality(diff)
print('차이값 정규성 pval =', round(diff_norm['pval'].iloc[0], 3), '→ 정규성 기각 못함 (대응 t 사용 가능)')

# pg.ttest(after, before, paired=True) — t·p·Cohen's d 를 한 표로
res_pair = pg.ttest(after, before, paired=True)
display(res_pair)
print('평균 변화(후-전) =', round(diff.mean(), 3), '초')
print('t =', round(res_pair['T'].iloc[0], 3), ', p =', format(res_pair['p_val'].iloc[0], '.6f'),
      ", Cohen's d =", round(res_pair['cohen_d'].iloc[0], 3), '→ 유의: 훈련 후 기록이 단축됨')

In [ ]:
# ③ 독립 2표본 t-검정: 일본산 vs 유럽산 연비 (서로 다른 차들)
japan_mpg = mpg[mpg['origin'] == 'japan']['mpg']
europe_mpg = mpg[mpg['origin'] == 'europe']['mpg']

# 등분산 여부로 Student(correction=False) / Welch(correction=True) 선택
two_je = mpg[mpg['origin'].isin(['japan', 'europe'])]
equal_var = bool(pg.homoscedasticity(data=two_je, dv='mpg', group='origin')['equal_var'].iloc[0])
print('등분산인가? equal_var =', equal_var, '→', '등분산(Student)' if equal_var else '이분산(Welch)')

# correction=not equal_var 로 자동 선택 (여기선 등분산 → Student). 표에 Cohen's d 포함
res_ind = pg.ttest(japan_mpg, europe_mpg, correction=not equal_var)
display(res_ind)
print('일본', round(japan_mpg.mean(), 2), 'vs 유럽', round(europe_mpg.mean(), 2),
      '/ t =', round(res_ind['T'].iloc[0], 3), ', p =', round(res_ind['p_val'].iloc[0], 4),
      "/ Cohen's d =", round(res_ind['cohen_d'].iloc[0], 3), '(중간 효과 — 유의하지만 차이 크기는 중간)')

### 🖐️ 함께 따라하기 — 만족도가 4.0이라는 주장 검정(1표본)

고객센터 팀장이 **"우리 센터의 평균 만족도는 4.0점"** 이라고 말합니다. 상담 600건의 `만족도` 로 이 주장을 1표본 t-검정으로 확인해 봅시다.

> 결과를 볼 때 **p-value 와 효과크기(Cohen's d)를 함께** 보세요. 이 둘이 서로 다른 이야기를 할 수 있습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) pg.ttest(cc['만족도'], 4.0) 으로 결과 표 res 를 만들어 display 한다
# 2) 표본 평균 만족도 cc['만족도'].mean() 을 소수 셋째 자리로 출력한다
# 3) res['T'](소수 셋째)와 res['p_val'](지수표기 format(p,'.3e'))를 출력한다
# 4) res['cohen_d'] 로 효과크기도 함께 확인한다
# 5) p 와 d 가 각각 무엇을 말하는지 생각해 본다 (유의성 vs 차이의 크기)

### ✅ 바로 확인 퀴즈

**1.** 같은 환자의 **투약 전·후 혈압**을 비교하려 합니다. 어떤 t-검정을 쓰나요?

<details><summary>정답 보기</summary>

**대응표본 t-검정**(`pg.ttest(after, before, paired=True)`)입니다. 같은 대상을 짝지어 재므로 개인차가 상쇄됩니다. 이때는 **차이값의 정규성**만 확인합니다.

</details>

**2.** 독립 2표본 t-검정 결과 `p = 0.016`, `Cohen's d = 0.40` 입니다. 어떻게 해석하나요?

<details><summary>정답 보기</summary>

`p < 0.05` 라 두 평균 차이가 **통계적으로 유의**하지만, `d ≈ 0.4` 는 기준(0.2/0.5/0.8) 상 **작음~중간** 사이입니다. "유의하다"(우연 아님)와 "크다"(효과크기)는 별개임을 보여 줍니다.

</details>

---
# 4. 모수 vs 비모수 — 정규성이 깨지면

## 왜 필요할까요?
정규성이 심하게 깨지거나(왜도 큼) 표본이 작고 이상치가 많으면, 평균 기반의 t-검정은 미덥지 않습니다. 이럴 때 값 대신 **순위(등수)** 로 비교하는 **비모수 검정**으로 갈아탑니다.

<img src="images/모수_vs_비모수.png" width="720">

| 상황(2집단) | 모수 | 비모수 대안 (Pingouin) |
|---|---|---|
| 독립 2집단 | 독립 t-검정 | **Mann-Whitney U**(`pg.mwu`) |
| 대응(전/후) | 대응 t-검정 | **Wilcoxon 부호순위**(`pg.wilcoxon`) |

- **Mann-Whitney U** 는 두 집단 값을 **한 줄로 순위 매긴 뒤** 순위합을 비교합니다. 이상치·비정규에 강건합니다.
- `pg.mwu` 는 결과 표에 **효과크기를 자동으로** 줍니다: **`RBC`**(rank-biserial, |r| ≈ 0.1 작음/0.3 중간/0.5 큼)와 **`CLES`**(공통언어 효과크기 — 한쪽에서 뽑은 값이 다른 쪽보다 클 확률). 🔧 대안: `scipy.stats.mannwhitneyu` (U·p만, 효과크기는 수동).

> 정규성이 만족되면 **모수 검정이 검정력이 더 높습니다**(같은 데이터로 더 잘 잡음). 비모수는 가정이 깨질 때의 안전한 대안입니다.

In [ ]:
# 미국산 연비는 정규성 위반(pg.normality 로 pval<0.001) → 순위 기반 비모수로 비교
usa_mpg = mpg[mpg['origin'] == 'usa']['mpg']
japan_mpg = mpg[mpg['origin'] == 'japan']['mpg']

# pg.mwu — U·p 와 함께 효과크기(RBC=rank-biserial, CLES)를 한 표로
mw = pg.mwu(usa_mpg, japan_mpg)
display(mw)
print('미국 중앙값 =', usa_mpg.median(), ' / 일본 중앙값 =', japan_mpg.median())
print('U =', mw['U_val'].iloc[0], ', p =', format(mw['p_val'].iloc[0], '.2e'), '→ 매우 유의: 연비 분포가 다름')
print('rank-biserial RBC =', round(mw['RBC'].iloc[0], 3), '  (|RBC|>0.5면 큰 효과)')
print('CLES =', round(mw['CLES'].iloc[0], 3), '  (미국차 연비가 일본차보다 클 확률)')

### 🖐️ 함께 따라하기 — 신규 고객이 더 오래 기다리나 (비모수)

앞에서 **대기시간은 정규성이 크게 깨진다**는 것을 확인했죠. 그러니 두 집단을 비교할 때 t-검정 대신 **Mann-Whitney U** 를 씁니다. 신규 고객과 기존 고객의 대기시간을 비교해 봅시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) new_wait, old_wait 를 cc['고객구분'] 이 '신규'/'기존' 인 행의 '대기시간_초' 로 만든다
# 2) pg.mwu(new_wait, old_wait) 로 결과 표 mw 를 만들어 display 한다
# 3) mw['U_val'] 와 mw['p_val'](지수표기 format(p,'.2e'))을 출력한다
# 4) mw['RBC'](rank-biserial 효과크기)를 소수 셋째 자리로 출력한다
# 5) 두 집단의 **중앙값**(median)을 각각 출력해 방향을 확인한다
#    (비모수 검정은 평균이 아니라 순위를 비교하므로 중앙값으로 설명하는 것이 자연스럽습니다)

### ✅ 바로 확인 퀴즈

**1.** 표본이 작고 이상치가 심해 정규성이 크게 깨졌습니다. 두 독립집단 비교에 무엇을 쓰나요?

<details><summary>정답 보기</summary>

**Mann-Whitney U 검정**(`pg.mwu`)입니다. 값 대신 **순위**로 비교하므로 이상치·비정규에 강건합니다. (대응표본이라면 Wilcoxon 부호순위 검정 `pg.wilcoxon` 을 씁니다.)

</details>

**2.** 정규성이 잘 만족되는데도 굳이 비모수 검정을 쓰면 손해가 있나요?

<details><summary>정답 보기</summary>

네. 가정이 만족되면 **모수 검정의 검정력이 더 높아** 같은 데이터로도 실제 차이를 더 잘 잡아냅니다. 비모수는 순위만 쓰느라 정보를 일부 버리므로, 가정이 깨질 때의 **안전한 대안**으로 씁니다.

</details>

## 짝이 있는 자료라면 — Wilcoxon 부호순위

비모수는 Mann-Whitney 하나가 아닙니다. **어떤 모수 검정을 대체하느냐**에 따라 짝이 정해져 있어요. 3절에서 배운 **대응 t** 의 비모수 짝이 **Wilcoxon 부호순위**(`pg.wilcoxon`)입니다.

| 상황 | 모수 검정 | 비모수 짝 |
|---|---|---|
| 독립 2집단 | 독립 t | Mann-Whitney U `pg.mwu` (위에서 실행) |
| 같은 대상 전/후 | 대응 t | **Wilcoxon 부호순위** `pg.wilcoxon` (아래) |

> 3집단 이상을 비교하는 비모수 검정(**Kruskal-Wallis**)도 있습니다. 다만 그건 **ANOVA 의 짝**이라, **5절에서 ANOVA 를 배운 뒤** 이어서 다룹니다.

> **가정이 만족돼도 한 번 더 돌려 보는 이유**: 아래 데이터는 정규성을 만족해 대응 t 가 적절합니다. 그런데도 비모수로 한 번 더 재 보는 것은 실무에서 흔한 **강건성 확인(robustness check)** 이에요. 두 방법이 **같은 결론**이면 그 결론을 안심하고 쓰고, **엇갈리면** 가정이나 이상치를 다시 들여다봅니다.

In [ ]:
# Wilcoxon 부호순위 — 대응 t 의 비모수 짝 (훈련 전/후 12명)
diff_train = train['after_sec'] - train['before_sec']
print('차이값 정규성 p =', round(pg.normality(diff_train)['pval'].iloc[0], 4),
      '→ 정규라서 대응 t 가 적절하지만, 비모수로도 확인해 본다')

pt = pg.ttest(train['before_sec'], train['after_sec'], paired=True)
wx = pg.wilcoxon(train['before_sec'], train['after_sec'])
display(wx)
print('대응 t      : t = %.3f, p = %.6f' % (pt['T'].iloc[0], pt['p_val'].iloc[0]))
fmt_w = 'Wilcoxon    : W = %.1f, p = %.6f, RBC = %.3f'
print(fmt_w % (wx['W_val'].iloc[0], wx['p_val'].iloc[0], wx['RBC'].iloc[0]))
print('중앙값 %.1f초 → %.1f초' % (train['before_sec'].median(), train['after_sec'].median()))
print('→ 두 방법 모두 p < 0.05. 같은 결론이니 \'훈련 후 시간이 줄었다\' 를 안심하고 말할 수 있다.')

### ✅ 바로 확인 퀴즈

**1.** 같은 환자의 복용 전·후 통증 점수를 비교하는데 차이값 정규성이 깨졌습니다. 무엇을 쓰나요?

<details><summary>정답 보기</summary>

**Wilcoxon 부호순위 검정**(`pg.wilcoxon`)입니다. 대응 t 의 비모수 짝이에요. 독립 2집단이 아니므로 Mann-Whitney 가 아니라는 점에 주의하세요 — **짝이 있는지부터** 보고 고릅니다.

</details>

**2.** 독립 2집단인데 정규성이 깨졌습니다. Wilcoxon 과 Mann-Whitney 중 무엇인가요?

<details><summary>정답 보기</summary>

**Mann-Whitney U**(`pg.mwu`)입니다. Wilcoxon 부호순위는 **짝이 있는(대응)** 자료용이에요. 비모수를 고를 때도 **짝이 있는지부터** 확인하는 것이 순서입니다.

</details>

---
# 5. 분산분석(ANOVA) — 3집단 이상의 평균 비교

## 왜 필요할까요?
집단이 3개 이상이면 t-검정을 여러 번? **안 됩니다.** 검정을 반복할수록 1종 오류가 **누적**되기 때문입니다. 3집단을 t-검정 3번(A-B, A-C, B-C)으로 하면 적어도 한 번 잘못 기각할 확률이 **1 − 0.95³ ≈ 14.3%** 로 부풀어 오릅니다. **ANOVA** 는 이를 **한 번의 검정**으로 해결합니다.

### 원리 — 변동을 쪼갠다
전체 흩어짐(SS_total)을 **집단 간 변동**(SS_between, 평균들이 서로 벌어진 정도)과 **집단 내 변동**(SS_within, 집단 안의 잡음)으로 나눕니다.

- **F = MS_between / MS_within** = (집단 간 신호) / (집단 내 잡음). F가 크면 "집단 차이가 잡음보다 크다" → 유의.
- H₀: "모든 집단 평균이 같다". 유의하면 "**적어도 하나**는 다르다"까지만 말합니다(어느 쌍인지는 사후검정).

<img src="images/ANOVA_변동분해.png" width="720">

### 📊 `pg.anova` 결과 표 읽는 법 — **`p_unc` 와 `np2` 두 개면 충분**
`detailed=True` 로 부르면 **두 줄짜리 표**가 나옵니다 — 위 줄이 **집단 간**(요인 이름), 아래 줄이 **집단 내**(`Within`). 결론은 **위 줄**에서 읽습니다.

| 열 | 중요도 | 뜻 |
|---|---|---|
| `p_unc` | **⭐** | **p-value** — `< 0.05` 면 "적어도 한 집단이 다르다" |
| `np2` | **⭐** | **효과크기 η²** — 집단 구분이 전체 변동의 몇 %를 설명하는가(0.01/0.06/0.14) |
| `F` | ○ | F 통계량 = 신호 ÷ 잡음. 클수록 유의(보고서에 함께 씀) |
| `Source`·`DF` | ○ | 변동의 출처(요인/`Within`)·자유도 |
| `SS`·`MS` | ✕ | F·η² 를 만드는 **중간 계산값**(원리 설명용) — 결론엔 안 씁니다 |

### 효과크기 η²(에타제곱)
**η² = SS_between / SS_total** — 집단 구분이 전체 변동의 몇 %를 설명하는지. 기준: **0.01 작음 / 0.06 중간 / 0.14 큼**. `pg.anova(..., detailed=True)` 결과의 **`np2` 열**이 이 값입니다(일원분산분석에서는 부분 η² = η²) — 손계산이 필요 없습니다. 아래 코드에서 **손계산 공식과 `np2` 가 일치하는지** 직접 확인해 봅니다.

### 사후검정(post-hoc) — 어느 쌍이 다른가
ANOVA가 유의하면 **Tukey HSD**(`pg.pairwise_tukey`)로 모든 쌍을 비교합니다. 다중검정 보정이 들어가 있어 t-검정 반복보다 안전하고, 결과 표에 쌍마다 **효과크기(`hedges`)까지** 나옵니다. 🔧 대안: `statsmodels`의 `pairwise_tukeyhsd`.

**📊 `pg.pairwise_tukey` 결과 표 읽는 법** — **한 줄이 한 쌍**입니다(3집단이면 3쌍 = 3줄).

| 열 | 중요도 | 뜻 |
|---|---|---|
| `p_tukey` | **⭐** | **그 쌍의 p-value**(다중검정 보정됨) — `< 0.05` 면 **그 두 집단은 유의하게 다름** |
| `hedges` | **⭐** | 그 쌍의 효과크기(0.2/0.5/0.8 기준) |
| `A`, `B`, `diff` | ○ | 비교한 두 집단과 평균 차이 — **`diff` 부호로 어느 쪽이 큰지** 읽습니다 |
| `mean_A`·`mean_B`·`se`·`T` | ✕ | 각 집단 평균·표준오차·통계량(참고값) |

> 즉 **`p_tukey < 0.05` 인 줄만 골라 읽으면** "어느 집단끼리 실제로 다른가"의 답이 됩니다.

<img src="images/사후검정_가이드.png" width="720">

> **가정이 다르면 분기**: 등분산이 깨지면 **Welch's ANOVA**(`pg.welch_anova`) → 사후 **Games-Howell**(`pg.pairwise_gameshowell`), 정규성이 깨지면 비모수 **Kruskal-Wallis**(`pg.kruskal`)로 갑니다. Pingouin 은 이 대안들도 **한 줄**이라, 아래 코드에서 One-way ANOVA + Tukey 를 주로 다루고 Welch·Games-Howell 도 바로 보여 드립니다.

In [ ]:
# 왜 ANOVA인가? 검정을 반복하면 1종 오류가 쌓인다 (FWER: 가족단위 오류율)
alpha = 0.05
print('검정 횟수별 FWER — 적어도 한 번 잘못 기각할 확률')
for k in [1, 3, 6, 10]:
    fwer = 1 - (1 - alpha) ** k
    print('  %2d번 검정 → %.1f%%' % (k, fwer * 100))
print()
print('한 번이면 5%로 묶이지만, 3번이면 약 14% · 6번이면 약 26%까지 부푼다.')
print('그래서 세 집단 이상은 t-검정을 반복하지 않고 ANOVA 로 한 번에 비교한다.')

In [ ]:
# 3집단(제조국별) 연비 평균 비교 — One-way ANOVA (pg.anova 하나로 F·p·η²)
# 등분산 확인: pg.homoscedasticity (긴 형태 — dv=mpg, group=origin)
lev = pg.homoscedasticity(data=mpg, dv='mpg', group='origin')
print('등분산 equal_var =', bool(lev['equal_var'].iloc[0]), '(True 면 One-way ANOVA 적절)')

# ANOVA: 결과 표의 F·p_unc·np2(=η²) 를 한 번에
aov = pg.anova(data=mpg, dv='mpg', between='origin', detailed=True)
display(aov)
f_stat = aov['F'].iloc[0]; p_val = aov['p_unc'].iloc[0]; eta_sq = aov['np2'].iloc[0]
print('F =', round(f_stat, 3), ', p =', format(p_val, '.2e'), '→ 유의: 적어도 한 집단의 평균이 다름')
print('η²(np2) =', round(eta_sq, 3), '  (0.14보다 큼 → 큰 효과: 제조국이 연비 변동의 약',
      round(eta_sq * 100), '%를 설명)')

In [ ]:
# (원리 확인) pg 의 np2 가 정말 η² = SS_between/SS_total 과 같은지 손계산으로 대조
grand_mean = mpg['mpg'].mean()
groups = [g['mpg'] for _, g in mpg.groupby('origin')]
ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
ss_total = ((mpg['mpg'] - grand_mean) ** 2).sum()
print('수동 η² =', round(ss_between / ss_total, 3), ' vs  pg.np2 =', round(eta_sq, 3), '→ 일치')

In [ ]:
# 사후검정 Tukey HSD: 어느 제조국 쌍이 다른지 (다중검정 보정 + 쌍별 효과크기 hedges)
tukey = pg.pairwise_tukey(data=mpg, dv='mpg', between='origin')
display(tukey)
print('p_tukey < 0.05 인 쌍은 평균이 유의하게 다름 (여기선 세 쌍 모두 다름)')
print('유의한 쌍 개수 :', (tukey['p_tukey'] < 0.05).sum())

# [참고] 등분산이 깨졌다면: Welch ANOVA → Games-Howell 사후검정 (pingouin 은 각각 한 줄)
display(pg.welch_anova(data=mpg, dv='mpg', between='origin'))
display(pg.pairwise_gameshowell(data=mpg, dv='mpg', between='origin')[['A', 'B', 'diff', 'pval', 'hedges']])

## 가정이 깨졌다면 — Kruskal-Wallis (ANOVA 의 비모수 짝)

4절에서 **독립 2집단**의 비모수 짝(Mann-Whitney)과 **대응표본**의 짝(Wilcoxon)을 봤습니다. 이제 ANOVA 를 배웠으니 마지막 짝을 채웁니다 — **3집단 이상**의 비모수 대안이 **Kruskal-Wallis**(`pg.kruskal`)입니다.

| 상황 | 모수 검정 | 비모수 짝 |
|---|---|---|
| 독립 2집단 | 독립 t | Mann-Whitney U `pg.mwu` |
| 같은 대상 전/후 | 대응 t | Wilcoxon 부호순위 `pg.wilcoxon` |
| **독립 3집단 이상** | **One-way ANOVA** | **Kruskal-Wallis** `pg.kruskal` |

평균 대신 **순위**를 쓰므로 정규성에 덜 민감합니다. 아래 데이터는 정규성을 만족해 ANOVA 가 적절하지만, **강건성 확인**으로 한 번 더 돌려 결론이 같은지 봅니다.

In [ ]:
# Kruskal-Wallis — ANOVA 의 비모수 짝 (원산지 3집단)
#   4절의 Mann-Whitney 는 미국 vs 일본 '두' 집단이었다. 유럽까지 '세' 집단이면 이쪽이다.
kw = pg.kruskal(data=mpg, dv='mpg', between='origin')
display(kw)
print('원산지별 연비 중앙값:')
print(mpg.groupby('origin')['mpg'].median().round(1).to_string())
fmt_k = 'H = %.2f, p = %s → 세 집단의 연비 분포가 같다고 보기 어렵다'
print(fmt_k % (kw['H'].iloc[0], format(kw['p_unc'].iloc[0], '.2e')))
print()
print('앞의 ANOVA 도 p 가 사실상 0 이었다 — 두 방법이 같은 결론이니 안심하고 쓸 수 있다.')
print('다만 Kruskal 도 ANOVA 처럼 \'어딘가 다르다\' 까지만 말한다.')

# 사후검정: 어느 쌍이 얼마나 다른가 — 쌍마다 pg.mwu 를 돌려 RBC(효과크기)를 읽는다
#   pg.kruskal 은 H·p 만 주고 효과크기를 주지 않는다. ANOVA 뒤 Tukey 가 하는 일을 여기서 이렇게 한다.
from itertools import combinations
print()
fmt_p = '  %-7s vs %-7s  p = %.3e, RBC = %+.3f'
for a, b in combinations(sorted(mpg['origin'].unique()), 2):
    r = pg.mwu(mpg[mpg['origin'] == a]['mpg'], mpg[mpg['origin'] == b]['mpg'])
    print(fmt_p % (a, b, r['p_val'].iloc[0], r['RBC'].iloc[0]))
print('RBC 해석 기준은 0.1 작음 / 0.3 중간 / 0.5 큼 — 일본 vs 미국이 가장 크게 갈린다.')
print('(쌍이 많으면 p 를 다중검정 보정해야 한다. 엄밀한 비모수 사후검정은 Dunn 검정 — 이 단원 범위 밖.)')

### ✅ 바로 확인 퀴즈

**1.** Kruskal-Wallis 로 p < 0.05 가 나왔습니다. "어느 집단끼리 다른지" 를 바로 알 수 있나요?

<details><summary>정답 보기</summary>

**알 수 없습니다.** ANOVA 와 마찬가지로 "적어도 한 쌍은 다르다" 까지만 말해 줍니다. 어느 쌍인지는 **사후검정**이 필요해요 — 모수라면 Tukey, 비모수라면 **Dunn 검정**입니다.

</details>

**2.** 3집단 비교에서 정규성이 깨졌고 등분산도 깨졌습니다. 무엇을 쓰나요?

<details><summary>정답 보기</summary>

**정규성이 깨진 것이 먼저**이므로 순위 기반 **Kruskal-Wallis** 로 갑니다. (정규성은 만족하는데 등분산만 깨졌다면 **Welch ANOVA** 였습니다 — 무엇이 깨졌는지에 따라 길이 갈립니다.)

</details>

### 🖐️ 함께 따라하기 — 채널에 따라 만족도가 다른가 (ANOVA + 사후검정)

상담 채널은 **전화·채팅·이메일** 세 가지입니다. 채널에 따라 `만족도` 평균이 다른지 검정해 봅시다.

이번엔 **등분산 점검부터** 하고, 그 결과에 따라 갈라집니다 — 앞에서 배운 분기를 실제로 적용해 보세요.

- 등분산이면 → `pg.anova` + `pg.pairwise_tukey`
- 등분산이 깨지면 → `pg.welch_anova` + `pg.pairwise_gameshowell`

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) pg.homoscedasticity(data=cc, dv='만족도', group='채널') 로 등분산을 먼저 점검해 display 한다
# 2) equal_var 값을 확인한다 (True 면 등분산, False 면 이분산)
# 3) pg.anova(data=cc, dv='만족도', between='채널', detailed=True) 로 aov 를 만들고
#    F(소수 셋째)·p_unc(지수표기)·np2(=η²)를 출력한다
# 4) 등분산이 깨졌다면 pg.welch_anova(data=cc, dv='만족도', between='채널') 도 함께 돌려 비교한다
# 5) 사후검정으로 어느 채널끼리 다른지 본다 — 이분산이면 pg.pairwise_gameshowell 을 쓴다
# 6) 채널별 평균 만족도도 출력해 방향을 확인한다

### ✅ 바로 확인 퀴즈

**1.** 4개 집단의 평균을 비교하려고 t-검정을 6번(모든 쌍) 했습니다. 무엇이 문제인가요?

<details><summary>정답 보기</summary>

검정을 반복할수록 **1종 오류가 누적**되어(다중비교 문제) 없는 차이를 "있다"고 할 확률이 커집니다. 그래서 먼저 **ANOVA** 로 한 번에 검정하고, 유의하면 **사후검정(Tukey HSD)** 으로 쌍을 봅니다.

</details>

**2.** ANOVA 결과 `F = 98.5, p < 0.001` 로 유의합니다. "세 집단이 모두 서로 다르다"고 말할 수 있나요?

<details><summary>정답 보기</summary>

아니요. ANOVA는 "**적어도 한 집단**이 다르다"까지만 말합니다. 어느 쌍이 다른지는 **Tukey HSD** 같은 사후검정으로 확인해야 합니다.

</details>

**3.** η² = 0.33 은 무엇을 뜻하나요?

<details><summary>정답 보기</summary>

집단 구분(제조국)이 연비 **전체 변동의 약 33%를 설명**한다는 뜻입니다. 기준 0.14를 크게 넘으니 **큰 효과**입니다.

</details>

---
# 6. 카이제곱 검정 — 범주형의 빈도를 검정한다

## 왜 필요할까요?
지금까지는 **수치형**(평균)이었습니다. 제조국·등급처럼 **범주형**은 평균이 없으니 **빈도(개수)** 로 검정합니다. 두 종류가 있습니다.

<img src="images/카이제곱_적합도_vs_독립성.png" width="720">

| 검정 | 질문 | 변수 개수 | 함수 | 자유도 |
|---|---|---|---|---|
| **적합도** | 관측 분포가 **기대(이론) 분포**와 맞나 | 1개 | `scipy.stats.chisquare(관측, f_exp=기대)` | k−1 |
| **독립성** | 두 범주 변수가 **서로 관련** 있나 | 2개 | `pg.chi2_independence(data, x, y)` | (r−1)(c−1) |

> 🔧 **도구 갈림**: **독립성**은 `pg.chi2_independence` 가 χ²·자유도·p 와 **효과크기 Cramér's V(`cramer`)를 한 표로** 줍니다(대안: `scipy.stats.chi2_contingency` + 수동 V). 반면 **적합도**는 Pingouin 에 없어 **`scipy.stats.chisquare`** 를 그대로 씁니다.

### 📊 `pg.chi2_independence` 결과 읽는 법 — **결과를 3개** 돌려줍니다
```python
expected, observed, stats_table = pg.chi2_independence(data=df, x='열1', y='열2')
```
- **`expected`** — **기대빈도** 표("두 변수가 무관하다면 이만큼"). 5 미만 칸이 많으면 카이제곱이 부정확(Cochran 규칙).
- **`observed`** — 실제 **관측빈도** 표.
- **`stats_table`** — 검정 결과 표. **여러 방식(test 열)이 줄로 나오는데, 우리가 쓰는 표준은 `pearson` 줄** 하나입니다.

| `stats_table` 의 열 | 중요도 | 뜻 |
|---|---|---|
| `pval` | **⭐** | **p-value** — `< 0.05` 면 두 범주가 **서로 관련 있음** |
| `cramer` | **⭐** | **Cramér's V** — 연관의 세기(0.1/0.3/0.5) |
| `test` | ○ | 검정 방식 — 여러 줄이 나오지만 **`pearson` 줄만 봅니다**(표준 카이제곱) |
| `chi2`·`dof` | ○ | χ² 통계량·자유도((행−1)×(열−1)) — 보고서에 함께 씀 |
| `lambda`·`power` | ✕ | 방식별 내부 모수·검정력 — 신경 쓰지 않아도 됩니다 |

> 그래서 코드에서 `stats_table[stats_table['test'] == 'pearson']` 로 그 줄을 골라 값을 꺼냅니다.

### 공통 원리
**χ² = Σ (관측 O − 기대 E)² / E** — 관측이 기대에서 멀수록 커집니다. χ²는 **오른쪽으로 치우친 분포**를 따르고, **오른쪽 꼬리 넓이 = p** 입니다.

<img src="images/카이제곱_분포.png" width="440">

- **기대빈도**: 독립성에서 Eᵢⱼ = (행 합 × 열 합) / 전체 — "두 변수가 무관하다면 이만큼" 이라는 가상 빈도.
- **Cochran 규칙**: 기대빈도 5 미만 칸이 20%를 넘거나 1 미만 칸이 있으면 카이제곱이 부정확 → 2×2는 **Fisher 정확검정**, R×C는 **범주 병합**.
- **사후분석(조정된 잔차)**: `statsmodels`의 `Table(...).standardized_resids` (🔧 Pingouin 미지원). |잔차| > 2 인 칸이 두드러진 칸(양수=기대보다 많음).
- **효과크기**: **Cramér's V**(`pg.chi2_independence` 의 `cramer`) `= √(χ² / (N·min(r−1, c−1)))`, 적합도는 **Cohen's w** `= √(χ²/N)`. 기준 0.1/0.3/0.5.

<img src="images/교차표_예시.png" width="440">

In [ ]:
# 적합도 검정(🔧 Pingouin 미지원 → scipy.stats.chisquare 사용): 제조국 분포가 '균등(1/3씩)'인가
observed = mpg['origin'].value_counts().sort_index()
n = len(mpg)
expected_uniform = np.array([n / 3] * 3)
print('관측 빈도  :', observed.to_dict())
print('기대 빈도  :', np.round(expected_uniform, 1).tolist(), '(균등 가정)')

chi2_stat, p_val = stats.chisquare(observed.values, f_exp=expected_uniform)
print('χ² =', round(chi2_stat, 2), ', 자유도 =', len(observed) - 1, ', p =', format(p_val, '.2e'))
print('→ 유의: 제조국 분포는 균등하지 않다 (미국산이 크게 많음)')

cohens_w = np.sqrt(chi2_stat / n)
print("Cohen's w =", round(cohens_w, 3), '  (큰 효과)')

In [ ]:
# 독립성 검정: 제조국(origin)과 연비등급이 관련 있나? — pg.chi2_independence 하나로 χ²·p·Cramér's V
# 연비등급은 중앙값 기준으로 파생 (중앙값 이상=고연비). pingouin 은 '열 이름'으로 지정하므로 컬럼으로 만든다
mpg['economy_grade'] = np.where(mpg['mpg'] >= mpg['mpg'].median(), '고연비', '저연비')
crosstab = pd.crosstab(mpg['origin'], mpg['economy_grade'])   # 표시·사후분석에 쓸 교차표
print('[교차표 — 관측빈도]')
display(crosstab)

expected, observed, chi_stats = pg.chi2_independence(data=mpg, x='origin', y='economy_grade')
pearson = chi_stats[chi_stats['test'] == 'pearson'].iloc[0]   # 표준 Pearson 카이제곱 행
chi2_stat, dof, p_val, cramers_v = pearson['chi2'], int(pearson['dof']), pearson['pval'], pearson['cramer']
print('χ² =', round(chi2_stat, 2), ', 자유도 =', dof, ', p =', format(p_val, '.2e'))
print('기대빈도 최솟값 =', round(expected.values.min(), 1), '(모두 5 이상 → Cochran 규칙 만족)')
print('→ 유의: 제조국과 연비등급은 서로 관련이 있다')
print("Cramér's V =", round(cramers_v, 3), '  (연관의 세기 — 큰 편, pg 가 자동 계산)')

In [ ]:
# 사후분석: 어느 칸이 기대보다 두드러지나 — 조정된 잔차(|잔차|>2면 유의)
# 🔧 조정된 잔차는 Pingouin 에 없어 statsmodels 의 Table 을 사용한다
from statsmodels.stats.contingency_tables import Table
adjusted = Table(crosstab.values).standardized_resids
resid_df = pd.DataFrame(adjusted, index=crosstab.index, columns=crosstab.columns)
print('[조정된 잔차 — 양수=기대보다 많음]')
display(resid_df.round(2))
print('예: 일본산·고연비 잔차가 크게 양수 → 일본산에 고연비가 유독 많다')

### 🖐️ 함께 따라하기 — 채널과 재문의는 관련 있나 (독립성)

실무에서 가장 많이 쓰는 것이 **독립성 검정**입니다(두 범주 변수가 서로 관련 있나). 상담 **채널**과 **재문의 여부**(`재문의여부`: 0=해결, 1=다시 문의)가 관련 있는지 검정해 봅시다. 관련이 있다면 "어떤 채널은 한 번에 해결이 잘 안 된다"는 뜻이 됩니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) pd.crosstab(cc['채널'], cc['재문의여부']) 로 교차표를 display 한다
# 2) pg.chi2_independence(data=cc, x='채널', y='재문의여부') 로 expected, observed, st 를 구한다
# 3) st 에서 test=='pearson' 행의 chi2·dof·pval·cramer 를 꺼내 출력한다
# 4) p<0.05 면 '채널과 재문의는 관련이 있다'고 판단한다
# 5) 채널별 재문의 **비율**도 구해(groupby 평균) 어느 채널이 문제인지 확인한다

### ✅ 바로 확인 퀴즈

**1.** "성별과 흡연 여부가 서로 관련 있는가"를 검정하려 합니다. 어떤 카이제곱 검정인가요?

<details><summary>정답 보기</summary>

두 범주 변수의 관련성을 보는 **독립성 검정**(`pg.chi2_independence`)입니다. 두 변수를 열 이름으로 넣으면 Cramér's V 까지 나옵니다. (변수 하나의 분포가 기대와 맞는지 보는 것은 **적합도 검정** — 이건 Pingouin 에 없어 `scipy.stats.chisquare` 를 씁니다.)

</details>

**2.** 카이제곱 독립성 검정에서 **기대빈도**는 어떻게 구하고 무엇을 뜻하나요?

<details><summary>정답 보기</summary>

각 칸의 기대빈도 = (그 행 합 × 그 열 합) / 전체 N 입니다. "두 변수가 **서로 무관**하다면 이 칸에 이만큼 있을 것"이라는 가상 빈도이고, 관측이 여기서 많이 벗어날수록 χ²가 커집니다.

</details>

**3.** 어떤 칸의 **조정된 잔차**가 +7.6 입니다. 어떻게 읽나요?

<details><summary>정답 보기</summary>

|잔차| > 2 라 유의하게 두드러진 칸이고, **부호가 양수**이므로 **기대보다 관측이 많다**는 뜻입니다(두 범주가 그 조합에서 유독 함께 나타남).

</details>

---
# 7. 검정 선택 가이드 — 무엇을 언제 쓰나

## 왜 필요할까요?
검정 종류가 많아 헷갈립니다. 하지만 **데이터 유형 두 가지**(비교 대상이 수치인가 범주인가, 집단이 몇 개인가)만 물으면 길이 정해집니다.

<img src="images/통계검정_종합_의사결정트리.png" width="760">

### 한눈에 보는 선택 규칙
- **한 집단의 평균 vs 기준값** → 정규성 OK: **1표본 t** / 위반: Wilcoxon 부호순위
- **같은 대상 전/후(짝)** → 차이값 정규성 OK: **대응 t** / 위반: **Wilcoxon**
- **독립 2집단 평균** → 정규성 OK & 등분산: **Student t** / 정규성 OK & 이분산: **Welch t** / 정규성 위반: **Mann-Whitney U**
- **독립 3집단+ 평균** → 정규성 OK & 등분산: **One-way ANOVA → Tukey** / 이분산: Welch ANOVA → Games-Howell / 정규성 위반: Kruskal-Wallis → Dunn
- **범주 1개의 분포 vs 이론** → **카이제곱 적합도**(기대빈도<5 많으면 대안)
- **범주 2개의 관련성** → **카이제곱 독립성**(2×2에서 기대빈도<5면 Fisher 정확검정)

> 그리고 검정 뒤엔 언제나 **효과크기**를 함께 봅니다 — "유의한가"와 "얼마나 큰가"는 다른 질문이니까요.

### 효과크기 한눈 정리 — 검정마다 이름이 다릅니다

| 비교 상황 | 모수 검정 · 효과크기 | 정규성이 깨지면 · 효과크기 |
|---|---|---|
| 두 집단 평균 | t-검정 · **Cohen's d** (0.2 / 0.5 / 0.8) | Mann-Whitney·Wilcoxon · **RBC** (0.1 / 0.3 / 0.5) |
| 세 집단 이상 평균 | ANOVA · **η²** (0.01 / 0.06 / 0.14) | Kruskal-Wallis · **사후검정에서 쌍별 RBC** (0.1 / 0.3 / 0.5) |
| 범주형 빈도 | 카이제곱 · **Cramér's V** (0.1 / 0.3 / 0.5) | 해당 없음(순위로 세울 값이 없음) |

> **Kruskal-Wallis 는 전체를 하나로 재는 효과크기를 `pg.kruskal` 이 주지 않습니다**(결과 표에 `H`·`p_unc` 뿐). 그래서 **어느 쌍이 얼마나 다른지**를 사후검정에서 쌍별로 봅니다 — **쌍마다 `pg.mwu` 를 돌려 `RBC`** 를 읽으면 됩니다. ANOVA 뒤에 Tukey 가 쌍별 `hedges` 를 주는 것과 같은 자리입니다.

> **비모수라고 다 같은 효과크기가 아닙니다.** RBC 는 **두 집단**을 견주는 값이라, 3집단 이상에서는 **쌍으로 나눠야** 쓸 수 있습니다.

> **Cramér's V 의 기준(0.1/0.3/0.5)은 자유도에 따라 달라집니다.** 표가 커서 자유도가 크면 **더 작은 값도 큰 효과**일 수 있으니, 기준을 기계적으로 적용하지 마세요.

### ✅ 바로 확인 퀴즈

**1.** "교육 방법 3가지(A/B/C)에 따라 시험 점수 평균이 다른가" — 점수는 정규·등분산입니다. 어떤 검정인가요?

<details><summary>정답 보기</summary>

3집단 이상 평균 비교이고 정규·등분산이 만족되므로 **One-way ANOVA** 입니다. 유의하면 **Tukey HSD** 로 어느 쌍이 다른지 봅니다.

</details>

**2.** "신약 복용 전·후 같은 환자의 통증 점수 차이" — 표본이 12명으로 작고 차이값이 심하게 비정규입니다. 무엇을 쓰나요?

<details><summary>정답 보기</summary>

짝지은 전/후이고 차이값 정규성이 깨졌으니 대응 t의 비모수 대안인 **Wilcoxon 부호순위 검정**을 씁니다.

</details>

---
## 🚀 응용 클론코딩 — 고객센터 개선안 검증 리포트

오늘 배운 것을 **한 흐름**으로 이어 봅시다: 가정 점검 → 검정 선택 → 검정 실행 → 효과크기 → 의사결정.

**미션**: 고객센터가 상담원 교육을 실시했습니다. 두 가지 질문에 답하는 짧은 리포트를 만듭니다.

1. **교육이 처리시간을 줄였는가?** — 같은 상담원의 교육 **전·후** 처리시간을 비교합니다(`callcenter_agents.csv`).
2. **어떤 채널을 먼저 개선해야 하는가?** — 채널별 만족도와 재문의율을 근거로 우선순위를 정합니다.

> 1번은 **같은 사람을 두 번 잰** 자료입니다. 독립 2표본이 아니라 **대응표본**이라는 점에 주의하세요. 대응표본에서는 **차이값의 정규성**만 확인하면 됩니다.

In [ ]:
# 🖐️ 함께 따라하기 — 개선안 검증 리포트 (아래 순서대로 직접 작성해 보세요)
# 1) pd.read_csv 로 data/callcenter_agents.csv 를 읽어 ag 에 담고 head() 로 훑어본다
#    (상담원 45명의 교육 전/후 평균 처리시간이 한 행에 짝지어 있다)
# 2) 차이값 diff = ag['교육후_처리시간_분'] - ag['교육전_처리시간_분'] 을 만들고
#    pg.normality(diff) 로 **차이값의 정규성**을 확인한다
# 3) 정규성이 만족되면 pg.ttest(교육후, 교육전, paired=True) 로 대응표본 t-검정을 한다
#    (정규성이 깨졌다면 pg.wilcoxon 이 대안이다)
# 4) t·p·cohen_d 와 평균 변화량을 출력해 '줄었는가/얼마나 줄었는가'에 답한다
# 5) 채널별 만족도 평균과 재문의 비율을 한 표로 만들어(groupby.agg) 개선 우선순위를 정한다
# 6) 위 근거로 '무엇을 어떻게 하겠다'는 결론 두세 문장을 print 한다

### ✅ 바로 확인 퀴즈

**1.** 교육 전·후 처리시간을 **독립 2표본** t-검정으로 비교하면 무엇이 문제인가요?

<details><summary>정답 보기</summary>

같은 상담원을 두 번 잰 자료인데 **짝을 무시**하게 됩니다. 대응표본으로 보면 개인차가 상쇄돼 **같은 데이터로도 차이를 더 잘 잡아냅니다**(검정력이 높아집니다). 짝이 있는 자료는 반드시 `paired=True` 로 보세요.

</details>

**2.** 이메일 채널의 만족도가 낮으니 "이메일을 없애면 만족도가 오른다"고 말해도 될까요?

<details><summary>정답 보기</summary>

**안 됩니다.** 관찰 데이터에서 얻은 관련성일 뿐 인과가 아닙니다. 복잡하고 오래 걸리는 문의가 애초에 이메일로 몰렸을 수도 있어요(선택 효과). 인과를 말하려면 **문의 유형을 통제**하거나 **실험(A/B)** 이 필요합니다.

</details>

---
## 이번 강의 정리

| 상황 | 검정 | Pingouin 함수 (효과크기 자동 포함) |
|---|---|---|
| 한 집단 vs 기준값 | 1표본 t | `pg.ttest(x, 기준값)` → `cohen_d` |
| 같은 대상 전/후 | 대응 t (비모수 Wilcoxon) | `pg.ttest(a, b, paired=True)` / `pg.wilcoxon` → `cohen_d` |
| 독립 2집단 | Student/Welch t (비모수 MWU) | `pg.ttest(a, b, correction=)` / `pg.mwu` → `cohen_d` / `RBC` |
| 독립 3집단+ | ANOVA → Tukey | `pg.anova` / `pg.pairwise_tukey` → `np2`(η²) / `hedges` |
| 범주 1개 분포 | 카이제곱 적합도 | `scipy.stats.chisquare` (🔧 pg 미지원) → Cohen's w |
| 범주 2개 관련성 | 카이제곱 독립성 | `pg.chi2_independence` → `cramer`(Cramér's V) |

- **가정 점검**(정규성 `pg.normality`+Q-Q, 등분산 `pg.homoscedasticity`)이 검정 선택을 좌우합니다.
- **Pingouin 의 최대 장점**: 검정 결과 표에 **효과크기·신뢰구간이 이미 들어 있어**, "유의성만 보고 효과크기를 빼먹는" 실수를 막아 줍니다.
- **p-value 는 우연 여부, 효과크기는 차이의 크기** — 둘을 늘 함께 봅니다.
- 지난 시간의 신뢰구간·분포·CLT가 이 모든 검정의 바탕이 되었습니다. (회귀는 Pingouin 이 아닌 `statsmodels` 로 — 다음 시간)

## ⏭️ 예고 — 다음 시간: 회귀분석(상관을 예측·설명으로)

이번 시간엔 **차이가 있는가**(검정)를 물었습니다. 다음 시간엔 한 걸음 더 나아가 **얼마나·어떻게 연결되는가**를 모델로 만듭니다.
- **상관 → 회귀** — 상관계수 r 의 제곱(**결정계수 r²**)이 곧 회귀의 설명력(R²)으로 이어집니다.
- **단순·다중 선형회귀** — `statsmodels` 로 회귀식을 세우고 계수·p-value·R² 를 읽습니다.
- **잔차 진단**과 **데이터 기반 의사결정(A/B 테스트)** 까지.

오늘 익힌 검정·효과크기·가정 점검이 회귀 계수의 유의성 판단에 그대로 쓰입니다. 수고하셨습니다!